# ⚽ FIFA World Cup 2026 — Live Dashboard

A **Spur App** that fuses two live data sources into one reactive dashboard:

- **Polymarket** (`gamma-api.polymarket.com`) — prediction-market *implied odds* for World Cup 2026 questions (winner, hosts, golden boot).
- **RSSHub** (`rsshub.app`, Google-News fallback) — the World Cup headline feed.

**Pipeline (reactive DAG):** two Python *source* cells pull each feed and publish an Arrow **port** (`wc_markets`, `wc_news`); the **Deno frontend cell** reads those ports and renders a **Perspective** datagrid + chart, with the live MCP tool `wc_snapshot` preferred when the plugin is running.

> Re-run a source cell (or arm a cron schedule on it) → the cascade re-renders the dashboard. In **App mode**, only the frontend cell is shown. Implied probabilities are *market prices*, not forecasts.

In [18]:
import os, sys

# Reuse the app's own data layer (server/worldcup.py) — the exact code the
# wc_markets / wc_snapshot MCP tools call. Single source of truth.
for _cand in (os.path.join(os.getcwd(), "server"),
              os.path.join(os.path.dirname(os.getcwd()), "server")):
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand)
        break
import worldcup

# --- Polymarket data source -------------------------------------------------
markets = worldcup.fetch_markets("world cup", 50)
live = bool(markets)
if not markets:
    # Offline / no active market yet — labelled sample so the dashboard renders.
    _sample = [
        ("[sample] Will Spain win the 2026 World Cup?",     "0.18", "4200000"),
        ("[sample] Will Argentina win the 2026 World Cup?", "0.15", "3900000"),
        ("[sample] Will France win the 2026 World Cup?",    "0.14", "3500000"),
        ("[sample] Will Brazil win the 2026 World Cup?",    "0.13", "3300000"),
        ("[sample] Will England win the 2026 World Cup?",   "0.11", "2600000"),
    ]
    markets = [
        worldcup.shape_market(
            {"question": q, "outcomes": '["Yes","No"]',
             "outcomePrices": f'["{p}","{round(1 - float(p), 2)}"]',
             "volume": v, "slug": ""}
        )
        for q, p, v in _sample
    ]

# Prove the Polymarket spur_rest DuckDB datasource is wired (cheap LIMIT 1 probe
# — never count(*), which would force the table function to paginate the whole
# active-market set).
try:
    import duckdb
    _ext = os.path.expanduser("~/.spur/extensions/spur_rest.duckdb_extension")
    _con = duckdb.connect(config={"allow_unsigned_extensions": "true"})
    _con.execute(f"LOAD '{_ext}'")
    _con.execute("SELECT question FROM polymarket_markets() LIMIT 1").fetchall()
    print("polymarket datasource OK: spur_rest table function reachable")
    _con.close()
except Exception as _e:  # noqa: BLE001
    print(f"(polymarket datasource probe skipped: {_e})")

spur.put("wc_markets", markets)
print(f"{len(markets)} World Cup markets ({'LIVE' if live else 'SAMPLE'}) -> port wc_markets")
markets[:3]

polymarket datasource OK: spur_rest table function reachable


question,outcome,yes_prob,implied_pct,volume,volume_24hr,liquidity,end_date,url
Will Ivory Coast reach the Semifinals at the 2026 FIFA World Cup?,Yes,0.0525,5.2,9917.900495999998,269.763124,38411.14382,2026-07-13,https://polymarket.com/market/will-ivory-coast-reach-the-semifinals-at-the-2026-fifa-world-cup-20260602145149727
Will 3+ matches be suspended by weather protocol during the 2026 FIFA World Cup?,Yes,0.51,51.0,997.532856,0.0,126.1479,2026-07-20,https://polymarket.com/market/will-3-matches-be-suspended-by-weather-protocol-during-the-2026-fifa-world-cup-20260610211249340
Will Romelu Lukaku record the most goal contributions at the 2026 FIFA World Cup?,Yes,0.007,0.7,996.340117,9.57425,6197.97492,2026-07-20,https://polymarket.com/market/will-romelu-lukaku-record-the-most-goal-contributions-at-the-2026-fifa-world-cup
Will Cape Verde reach the Round of 16 at the 2026 FIFA World Cup?,Yes,0.059,5.9,993.581064,74.35788,8112.00098,2026-07-04,https://polymarket.com/market/will-cape-verde-reach-the-round-of-16-at-the-2026-fifa-world-cup-20260602025120766
Will Mohamed Salah record the most assists at the 2026 FIFA World Cup?,Yes,0.0055,0.5,993.38,0.0,9873.6097,2026-08-03,https://polymarket.com/market/will-mohamed-salah-record-the-most-assists-at-the-2026-fifa-world-cup


8 World Cup markets (LIVE) -> port wc_markets


[{'question': 'Will Ivory Coast reach the Semifinals at the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.0525,
  'implied_pct': 5.2,
  'volume': 9917.900495999998,
  'volume_24hr': 269.763124,
  'liquidity': 38411.14382,
  'end_date': '2026-07-13',
  'url': 'https://polymarket.com/market/will-ivory-coast-reach-the-semifinals-at-the-2026-fifa-world-cup-20260602145149727'},
 {'question': 'Will 3+ matches be suspended by weather protocol during the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.51,
  'implied_pct': 51.0,
  'volume': 997.532856,
  'volume_24hr': 0.0,
  'liquidity': 126.1479,
  'end_date': '2026-07-20',
  'url': 'https://polymarket.com/market/will-3-matches-be-suspended-by-weather-protocol-during-the-2026-fifa-world-cup-20260610211249340'},
 {'question': 'Will Romelu Lukaku record the most goal contributions at the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.007,
  'implied_pct': 0.7,
  'volume': 996.340117,
  'volume_24hr': 9.57425,

In [19]:
import os, sys

for _cand in (os.path.join(os.getcwd(), "server"),
              os.path.join(os.path.dirname(os.getcwd()), "server")):
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand)
        break
import worldcup

# --- RSSHub data source (rsshub.app first, Google-News RSS fallback) --------
news = worldcup.fetch_news(30)
live_news = bool(news)
if not news:
    news = [
        {"title": "[sample] World Cup 2026 host cities finalize match schedule",
         "link": "https://example.org/wc/1", "published": "", "source": "sample"},
        {"title": "[sample] Qualification race tightens across confederations",
         "link": "https://example.org/wc/2", "published": "", "source": "sample"},
        {"title": "[sample] Ticket demand sets a record for the opening match",
         "link": "https://example.org/wc/3", "published": "", "source": "sample"},
    ]

spur.put("wc_news", news)
_src = news[0]["source"] if news else "none"
print(f"{len(news)} headlines ({'LIVE via ' + _src if live_news else 'SAMPLE'}) -> port wc_news")
news[:3]

title,link,published,source
World Cup 2026: Fifa urged to remove official over hand gesture; teams hit back at Ceferin; Iran arrive in US – as it happened - The Guardian,https://news.google.com/rss/articles/CBMi4AFBVV95cUxPQjI2RmstSjktUUZmZTZWM1RraUR3bE94NkNLV0I3blBHenZBUllUUGctWXFtdlRfeWRRbVNobG5NLUgxRUlJUE1OZnc5cVJpdjM4YlQ2d3pPLWVhcTFIelVxOHRheW5Odm1Tem1CLV9zQnd0blo0THB6S3FhUF9nQ041WkhIbzI0blFCZHl2LWRfUHI4a0J5Qnd1aG5pYURNc0U0bkQ0MzhuOWVYUm12M3d4bi1FVzByRjlCalZrV3lLaXN0WXNBX2VuZzhHVGFjX1Y1XzNzZTNEMmY5aUhqYw?oc=5,"Mon, 15 Jun 2026 14:59:10 GMT",rss
Belgium vs Egypt live updates: World Cup 2026 game latest news and buildup - The New York Times,https://news.google.com/rss/articles/CBMitgFBVV95cUxQZHBVXzhFTW5fQ3Ezcl9Tc1NSbHdTZWpRRExSb2xBOWpPNzNNMU9OMThPZXZPTUdhNm12NUxNNWhMUTJ3eGt3X0RzRGJYSzhpWlVBb0JBZmpVemhYNXl4Tk12NlpVOFFrZGFsOTROcV83SXNxX3AySnNweHVjVDVqMm5wWUZndlZ4YjdWUWtDcGp1T2lvZ25kYk1yUVotTV9zZ3Y1T0s4ZGc2ajAwU09xN1hWM2MyUQ?oc=5,"Mon, 15 Jun 2026 16:45:00 GMT",rss
World Cup 2026: Shaun Evans - Fifa seeks explanation over VAR official's hand gesture - BBC,https://news.google.com/rss/articles/CBMiZ0FVX3lxTE9IU3VxYlkwb3hhMWEwMEJ0S2U1V1c0YVA5VzJFZHh3ZmEtRVN2d1oxMjcxTXQ3bFhKakZQMWFVN1NaaG53cmxkaU43QlJtQzJ3NEtURHh3R2pOTzVlQkZtWWdId2lJTm8?oc=5,"Mon, 15 Jun 2026 12:59:09 GMT",rss
Spain vs Cape Verde live: World Cup 2026 - Al Jazeera,https://news.google.com/rss/articles/CBMilgFBVV95cUxQZXhxSHFBaTZEb2JpZWZoNUlBMldHY3ZZdE9MSllVakhUZGRfQVNBTkVSRjZSTnQ2TkdldHpRSWN0WGEyMUU2SHZRVUdZdE9ldXVEbWFnbEJWWVV4U29aZEstZTBSX280Q2pDOEVvWWxzeFk1a3hDZ21ScXMwSFdtR3M2QmQxdW1uWDA4dWdSNWFwaS1WZUHSAZsBQVVfeXFMTnZVRUFPNzhBR0dEbnhnMVBRUjhqdkRuV2ZIUUhJSGkwc19RdVZlYVg5WmV0TkRfM19vb0RmYmR3dFRsTXp5WDhCMWZkUU81LTRsbGdRNzJRR2tZVVFQbFpURmwyc1FNNHFzanNJQVN5T1VnVEVMdEg3NTA3eGxhMng2cy12VWp5N2FTaFZodUxqdDc5SU1iRDZOZW8?oc=5,"Mon, 15 Jun 2026 16:52:30 GMT",rss
"World Cup 2026: guide to all 1,248 players - The Guardian",https://news.google.com/rss/articles/CBMiogFBVV95cUxNclJpaTBTTU1nZUlXRHJ4Q01RbXM2RmlqQVZBaDRrczhHeWhMVGh5bkVabUhjbDVzUS1UQXZGQUx5V2ZtQU02NXdEb1QyV0xsOVkwYXdWc3NJZkdkdnNtdVZFU0JYM3dBaEcwUEdCZW9xYUpmM1QxNVNKcUQzbTIwODRJaVotcHRGM0E5WFo3NVJMMklvTmNSUXRVc1JkSlhtMmc?oc=5,"Fri, 05 Jun 2026 07:00:00 GMT",rss


30 headlines (LIVE via rss) -> port wc_news


[{'title': 'World Cup 2026: Fifa urged to remove official over hand gesture; teams hit back at Ceferin; Iran arrive in US – as it happened - The Guardian',
  'link': 'https://news.google.com/rss/articles/CBMi4AFBVV95cUxPQjI2RmstSjktUUZmZTZWM1RraUR3bE94NkNLV0I3blBHenZBUllUUGctWXFtdlRfeWRRbVNobG5NLUgxRUlJUE1OZnc5cVJpdjM4YlQ2d3pPLWVhcTFIelVxOHRheW5Odm1Tem1CLV9zQnd0blo0THB6S3FhUF9nQ041WkhIbzI0blFCZHl2LWRfUHI4a0J5Qnd1aG5pYURNc0U0bkQ0MzhuOWVYUm12M3d4bi1FVzByRjlCalZrV3lLaXN0WXNBX2VuZzhHVGFjX1Y1XzNzZTNEMmY5aUhqYw?oc=5',
  'published': 'Mon, 15 Jun 2026 14:59:10 GMT',
  'source': 'rss'},
 {'title': 'Belgium vs Egypt live updates: World Cup 2026 game latest news and buildup - The New York Times',
  'link': 'https://news.google.com/rss/articles/CBMitgFBVV95cUxQZHBVXzhFTW5fQ3Ezcl9Tc1NSbHdTZWpRRExSb2xBOWpPNzNNMU9OMThPZXZPTUdhNm12NUxNNWhMUTJ3eGt3X0RzRGJYSzhpWlVBb0JBZmpVemhYNXl4Tk12NlpVOFFrZGFsOTROcV83SXNxX3AySnNweHVjVDVqMm5wWUZndlZ4YjdWUWtDcGp1T2lvZ25kYk1yUVotTV9zZ3Y1T0s4ZGc2ajAwU09xN1hWM2MyUQ?oc=5'

In [19]:
// World Cup 2026 — Perspective dashboard (frontend cell).
// Renders purely from the wc_markets / wc_news Arrow ports, which the two
// source cells populate reactively. To refresh, re-run a source cell (or arm a
// cron schedule on it) — the cascade re-renders this cell. We deliberately do
// NOT call an MCP tool here: that would put a blocking network + app-plugin
// round-trip in the App-mode render path (slow on open, can stall if the
// plugin is still spawning). The ports already hold the data.

function rowsOf(t) { try { return t && t.toArray ? t.toArray() : []; } catch (_) { return []; } }
function coerceMarket(r) {
  return {
    question: String(r.question ?? ""),
    implied_pct: r.implied_pct == null ? null : Number(r.implied_pct),
    volume: Number(r.volume ?? 0),
    end_date: String(r.end_date ?? ""),
    url: String(r.url ?? ""),
  };
}
function coerceNews(r) {
  return {
    title: String(r.title ?? ""),
    link: String(r.link ?? ""),
    published: String(r.published ?? ""),
    source: String(r.source ?? ""),
  };
}

const markets = rowsOf(spur.get("wc_markets")).map(coerceMarket);
const news = rowsOf(spur.get("wc_news")).map(coerceNews);

const priced = markets.filter((m) => m.implied_pct != null);
const favorite = priced.slice().sort((a, b) => b.implied_pct - a.implied_pct)[0] || null;
const totalVol = markets.reduce((s, m) => s + (m.volume || 0), 0);
const isSample = markets.some((m) => m.question.startsWith("[sample]")) || news.some((n) => n.source === "sample");

const esc = (s) => String(s == null ? "" : s).replace(/[&<>"]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
const fmtVol = (v) => "$" + (Number(v) || 0).toLocaleString("en-US", { maximumFractionDigits: 0 });
const fmtPct = (v) => (v == null ? "—" : Number(v).toFixed(1) + "%");

const kpiCards = [
  ["Market favorite", favorite ? esc(favorite.question.replace(/^\[sample\]\s*/, "")) : "—", favorite ? fmtPct(favorite.implied_pct) : ""],
  ["Markets tracked", String(markets.length), ""],
  ["Total volume", fmtVol(totalVol), ""],
  ["Headlines", String(news.length), ""],
].map(([label, big, sub]) => `<div class="kpi"><div class="kpi-label">${label}</div><div class="kpi-big">${big}</div><div class="kpi-sub">${sub}</div></div>`).join("");

const newsItems = news.slice(0, 18).map((n) => `<li><a href="${esc(n.link)}" target="_blank" rel="noopener">${esc(n.title)}</a><span class="src">${esc(n.source)}</span></li>`).join("") || '<li class="muted">No headlines.</li>';

// Browser-side module script. NOTE: no backticks / no ${} inside INNER so the
// outer template literal does not interpolate it.
const INNER = `
const fmtPct = (v) => (v == null ? "—" : Number(v).toFixed(1) + "%");
const fmtVol = (v) => "$" + (Number(v) || 0).toLocaleString("en-US", { maximumFractionDigits: 0 });
const esc = (s) => String(s == null ? "" : s).replace(/[&<>]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;" }[c]));
function fallbackTable(rows) {
  let h = "<table class='tbl'><thead><tr><th>Market</th><th class='num'>Implied</th><th class='num'>Volume</th><th>Ends</th></tr></thead><tbody>";
  for (const r of rows) { h += "<tr><td>" + esc(r.question) + "</td><td class='num'>" + fmtPct(r.implied_pct) + "</td><td class='num'>" + fmtVol(r.volume) + "</td><td>" + esc(r.end_date || "") + "</td></tr>"; }
  return h + "</tbody></table>";
}
const container = document.getElementById("pv");
const data = MARKETS.length ? MARKETS : [{ question: "(no markets)", implied_pct: null, volume: 0, end_date: "", url: "" }];
let ok = false;
for (const v of ["3.7.0", "3.3.0"]) {
  try {
    const base = "https://cdn.jsdelivr.net/npm/@finos/";
    const perspective = (await import(base + "perspective@" + v + "/dist/cdn/perspective.js")).default;
    await import(base + "perspective-viewer@" + v + "/dist/cdn/perspective-viewer.js");
    await import(base + "perspective-viewer-datagrid@" + v + "/dist/cdn/perspective-viewer-datagrid.js");
    await import(base + "perspective-viewer-d3fc@" + v + "/dist/cdn/perspective-viewer-d3fc.js");
    const worker = await perspective.worker();
    const table = await worker.table(data);
    const viewer = document.createElement("perspective-viewer");
    container.innerHTML = "";
    container.appendChild(viewer);
    await viewer.load(table);
    await viewer.restore({ plugin: "Datagrid", columns: ["question", "implied_pct", "volume", "end_date"], sort: [["volume", "desc"]], theme: "Pro Dark" });
    ok = true;
    break;
  } catch (e) { /* try next version */ }
}
if (!ok) { container.classList.add("fallback"); container.innerHTML = fallbackTable(data); }
`;

const safeMarkets = JSON.stringify(markets).replace(/</g, "\\u003c");
const scriptTag = '<scr' + 'ipt type="module">\nconst MARKETS = ' + safeMarkets + ';\n' + INNER + '\n</scr' + 'ipt>';

const html = `<!doctype html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@finos/perspective-viewer@3.7.0/dist/css/themes.css">
<style>
  :root { color-scheme: dark; }
  * { box-sizing: border-box; }
  body { margin: 0; background: #0a0e1a; color: #e6edf3; font-family: Inter, system-ui, sans-serif; }
  .wrap { max-width: 1180px; margin: 0 auto; padding: 22px; }
  .top { display: flex; align-items: baseline; gap: 12px; flex-wrap: wrap; }
  h1 { font-size: 22px; margin: 0; }
  .badge { font-size: 11px; font-weight: 700; padding: 3px 9px; border-radius: 999px; letter-spacing: .04em; }
  .badge.live { background: #0f3d2e; color: #34d399; }
  .badge.sample { background: #3d2f0f; color: #fbbf24; }
  .sub { color: #93a1b3; font-size: 13px; margin: 4px 0 18px; }
  .kpis { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin-bottom: 18px; }
  .kpi { background: #121829; border: 1px solid #1e2740; border-radius: 14px; padding: 14px 16px; }
  .kpi-label { color: #93a1b3; font-size: 12px; text-transform: uppercase; letter-spacing: .05em; }
  .kpi-big { font-size: 19px; font-weight: 700; margin-top: 6px; line-height: 1.25; }
  .kpi-sub { color: #34d399; font-weight: 700; margin-top: 2px; }
  .grid { display: grid; grid-template-columns: 1.7fr 1fr; gap: 16px; align-items: start; }
  .card { background: #121829; border: 1px solid #1e2740; border-radius: 14px; overflow: hidden; }
  .card h2 { font-size: 13px; text-transform: uppercase; letter-spacing: .06em; color: #93a1b3; margin: 0; padding: 14px 16px; border-bottom: 1px solid #1e2740; }
  #pv { height: 460px; width: 100%; }
  #pv.fallback { height: auto; max-height: 460px; overflow: auto; }
  .tbl { width: 100%; border-collapse: collapse; font-size: 13px; }
  .tbl th, .tbl td { text-align: left; padding: 8px 12px; border-bottom: 1px solid #1a2236; }
  .tbl th { color: #93a1b3; font-weight: 600; }
  .tbl td.num, .tbl th.num { text-align: right; font-variant-numeric: tabular-nums; }
  ul.news { list-style: none; margin: 0; padding: 4px 0; max-height: 460px; overflow: auto; }
  ul.news li { padding: 9px 16px; border-bottom: 1px solid #1a2236; font-size: 13px; display: flex; justify-content: space-between; gap: 10px; }
  ul.news a { color: #cdd9e5; text-decoration: none; }
  ul.news a:hover { color: #58a6ff; text-decoration: underline; }
  ul.news .src { color: #5b6b82; font-size: 11px; text-transform: uppercase; flex: none; }
  .muted { color: #5b6b82; }
  footer { color: #5b6b82; font-size: 11px; margin-top: 16px; }
</style></head>
<body><div class="wrap">
  <div class="top"><h1>⚽ FIFA World Cup 2026 — Live Dashboard</h1><span class="badge ${isSample ? "sample" : "live"}">${isSample ? "SAMPLE DATA" : "LIVE"}</span></div>
  <div class="sub">Polymarket implied odds × RSSHub headlines · prediction-market prices, not forecasts</div>
  <div class="kpis">${kpiCards}</div>
  <div class="grid">
    <div class="card"><h2>Markets — implied probability &amp; volume (Perspective)</h2><div id="pv">Loading Perspective…</div></div>
    <div class="card"><h2>Latest headlines</h2><ul class="news">${newsItems}</ul></div>
  </div>
  <footer>Sources: gamma-api.polymarket.com · rsshub.app (Google-News fallback). Re-run a source cell to refresh. Rendered by the world-cup-2026 Spur App.</footer>
</div>${scriptTag}</body></html>`;

await Deno.jupyter.display({ "text/html": html }, { raw: true });

<!doctype html> 
 
 
 
 ⚽ FIFA World Cup 2026 — Live Dashboard LIVE 
 Polymarket implied odds × RSSHub headlines · prediction-market prices, not forecasts 
 Market favorite Will 3+ matches be suspended by weather protocol during the 2026 FIFA World Cup? 51.0% Markets tracked 8 Total volume $15,976 Headlines 30 
 
 Markets — implied probability & volume (Perspective) Loading Perspective… 
 Latest headlines World Cup 2026: Fifa urged to remove official over hand gesture; teams hit back at Ceferin; Iran arrive in US – as it happened - The Guardian rss Belgium vs Egypt live updates: World Cup 2026 game latest news and buildup - The New York Times rss World Cup 2026: Shaun Evans - Fifa seeks explanation over VAR official's hand gesture - BBC rss Spain vs Cape Verde live: World Cup 2026 - Al Jazeera rss World Cup 2026: guide to all 1,248 players - The Guardian rss A team-by-team guide to the 2026 World Cup: What to expect and who to watch - The Athletic - The New York Times rss World Cup 2026: a visual guide to the stadiums across the trio of host nations - The Guardian rss Exclusive: Iran supreme leader’s adviser says talks deadlocked over $24 billion and warns of wider war - CNN rss "Don't know what else we could've done:" David Malukas' post-race interview after heartbreaking INDY 500 loss - FOX Sports rss Cape Verde's national anthem is heard for the first time ever at a FIFA World Cup 🇨🇻 - FOX Sports rss From The Sports Desk: One day after peace agreement, Iran takes field for World Cup - NBC News rss It's been a thrilling start to the World Cup. Here are the highlights and what's next - NPR rss 'Finally': Norway star Erling Haaland on fulfilling World Cup dream - ESPN rss Why Lamine Yamal Isn’t Playing for Spain vs. Cabo Verde—2026 World Cup Opener - Sports Illustrated rss Netherlands 2-2 Japan - FIFA rss When is Cape Verde's next World Cup game? Shop tickets now - USA Today rss New Zealand’s diplomatic breakaway - Politico rss FIFA World Cup 2026: Here is the June 15 schedule for Day 5 of group matches - WPLG Local 10 rss 
 
 Sources: gamma-api.polymarket.com · rsshub.app (Google-News fallback). Re-run a source cell to refresh. Rendered by the world-cup-2026 Spur App.